# 기본 챗봇과 체인 — 주피터 따라 하기
이 노트북은 **완성된 코드**입니다. 위에서부터 한 셀씩 실행하며 같은 메모가 **프롬프트 → 모델 → 파서**를 지날 때마다 결과의 **종류**가 어떻게 바뀌는지 확인합니다.

- 모든 셀은 ▶ 제공 코드입니다. 빈칸이나 채점 셀은 없습니다.
- 직접 구현하는 과제는 별도 런박스에서 합니다. 같은 폴더의 `바이브코딩_대상별전달문/실습_대상별전달문_바이브코딩.html`을 여세요.

이 노트북은 **독립 Runbox용**입니다. 다른 노트북을 열지 않아도 필요한 코드가 모두 들어 있습니다.

### 0. 실습 환경 확인
Python 버전이 **3.12.x**(3.12로 시작)로 나오면 정상입니다.

In [2]:
%pip install langchain langchain-core langchain-openai langgraph

  Using cached langgraph-1.2.24-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.14.4-py3-none-any.whl.metadata (143 kB)
  Using cached certifi-2026.11.20-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.48-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_protocol-0.0.22-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.16.2-py3-none-any.whl.metadata (40 kB)
  Using cached tenacity-9.2.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached langgraph_checkpoint-4.3.2-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached langgraph_sdk-0.4.14-py3-none-any.whl.metadata (5.1 kB)
  Using cached xxhash-4.1.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (18 kB)
  Usin

Using cached jsonpointer-3.12-py3-none-any.whl (10 kB)
Using cached langchain_protocol-0.0.22-py3-none-any.whl (7.6 kB)
Using cached openai-3.59.0-py3-none-any.whl (2.5 MB)
Using cached jiter-0.19.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (338 kB)
Using cached orjson-3.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (130 kB)
Using cached ormsgpack-1.13.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (273 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.3-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl (257 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached tenacity-9.2.1-py3-none-any.whl (32 kB)
Using cached tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl (1.2 MB)
Using cached truststore-0.10.5-py3-none-any.whl (19 kB)
Using cached typing_inspection-0.4.5-py3-none-any.whl (14 

In [3]:
# ▶ 제공 코드 — 실습 환경(커널) 확인
import sys
from importlib.metadata import version

print("Python", sys.version.split()[0], "(기준 3.12.x)")
for name in ["langchain", "langchain-core", "langchain-openai", "langgraph"]:
    print(f"{name:18s}", version(name))

Python 3.12.9 (기준 3.12.x)
langchain          1.5.2
langchain-core     1.8.5
langchain-openai   1.8.5
langgraph          1.2.24


### 1. 설정 셀 — 모델 연결
안내받은 **수업용 API 키**(`API_KEY`)와 **채팅 모델 주소**(`BASE_URL`)를 따옴표 안에 직접 입력하세요. 모델은 `gpt-6-luna`로 정해져 있어 바꾸지 않습니다. 이 실습은 채팅 모델만 쓰므로 임베딩 설정은 없습니다.
- 두 값 중 하나라도 자리표시자로 남아 있으면 이 셀이 모델을 호출하기 전에 멈추고 무엇을 입력할지 알려 줍니다.
- `.env` 파일은 사용하지 않습니다.
- 이 셀은 AI 채팅창이나 AI 코딩 도우미에 **붙여넣지 마세요**. 노트북을 제출·공유하기 전에는 키를 다시 `"여기에_수업용_API_키_입력"`으로 되돌립니다.

In [4]:
# ▶ 제공 코드 — 설정 셀 (따옴표 안의 값만 바꾸세요)
API_KEY = ""
BASE_URL = ""   # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                  # 수업에서 쓰는 채팅·도구 호출 모델(확정값, 바꾸지 않음)

missing = [name for name, value in (("API_KEY", API_KEY), ("BASE_URL", BASE_URL)) if value.startswith("여기에")]
if missing:
    raise ValueError(", ".join(missing) + "에 안내받은 값을 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL)
print("모델 준비 완료:", MODEL_NAME)

모델 준비 완료: gpt-6-luna


### 2. 첫 모델 호출 기준 코드
첫 모델 호출 실습의 세 줄 인계 규칙과 메모를 그대로 가져왔습니다. 메시지를 직접 만들어 `llm.invoke()`를 부르는 방식입니다.

In [5]:
# ▶ 제공 코드 — 첫 모델 호출 기준 코드 (가상 메모 · 인계 규칙 · make_handover)
memo = """A라인 온도 센서 교체 완료.
14시 재가동 예정.
재가동 후 30분 동안 온도 로그 확인 필요."""

HANDOVER_RULES = """당신은 제조 현장의 교대 인계 도우미입니다.
메모를 정확히 세 줄로만 정리하세요.
1줄: '완료: '로 시작하고 끝난 작업을 씁니다.
2줄: '예정: '로 시작하고 앞으로 할 작업과 시각을 씁니다.
3줄: '확인: '로 시작하고 다음 근무자가 확인할 일을 씁니다.
메모에 없는 내용은 만들지 말고, 해당 내용이 없으면 '기재 없음'이라고 씁니다.
'예정'인 일을 '완료'로 바꾸지 마세요."""

from langchain.messages import SystemMessage, HumanMessage

def make_handover(memo: str) -> str:
    """메모를 완료·예정·확인 세 줄 인계문으로 바꿉니다."""
    messages = [SystemMessage(HANDOVER_RULES), HumanMessage(memo)]
    return llm.invoke(messages).content

print(make_handover(memo))

완료: A라인 온도 센서 교체
예정: 14시 재가동
확인: 재가동 후 30분 동안 온도 로그 확인


### 3. 부품 ① 프롬프트 템플릿 — `ChatPromptTemplate`
`{memo}` 자리에 값을 끼워 넣는 서식입니다. **아직 모델을 호출하지 않았습니다.** 모델에 보낼 지시와 메모를 준비했을 뿐이고, 결과는 역할별 메시지를 담은 `ChatPromptValue`입니다.

In [6]:
# ▶ 제공 코드 — 프롬프트 템플릿만 실행
from langchain_core.prompts import ChatPromptTemplate

handover_prompt = ChatPromptTemplate.from_messages([
    ("system", HANDOVER_RULES),
    ("human", "{memo}"),
])
prompt_value = handover_prompt.invoke({"memo": memo})

print("종류:", type(prompt_value).__name__)
for m in prompt_value.to_messages():
    print(f"[{m.type}]", m.content[:60], "...\n" if len(m.content) > 60 else "\n")

종류: ChatPromptValue
[system] 당신은 제조 현장의 교대 인계 도우미입니다.
메모를 정확히 세 줄로만 정리하세요.
1줄: '완료: '로 시작 ...

[human] A라인 온도 센서 교체 완료.
14시 재가동 예정.
재가동 후 30분 동안 온도 로그 확인 필요. 



### 4. 부품 ② 모델
프롬프트가 만든 메시지를 그대로 모델에 넣습니다. 결과는 모델이 반환한 답변 메시지 **AIMessage**입니다. 이 셀에서 모델을 한 번 호출합니다.

In [7]:
# ▶ 제공 코드 — 모델만 실행
ai_message = llm.invoke(prompt_value)
print("종류:", type(ai_message).__name__)
print(ai_message.content)

종류: AIMessage
완료: A라인 온도 센서 교체
예정: 14시 재가동
확인: 재가동 후 30분 동안 온도 로그 확인


### 5. 부품 ③ 출력 파서 — `StrOutputParser`
AIMessage에서 **답변 본문을 문자열(str)로** 꺼냅니다. 줄 수 같은 형식은 프롬프트가 요청하고, 파서는 본문을 꺼낼 뿐입니다. 모델이 네 줄을 쓰면 파서를 거쳐도 네 줄입니다.

In [8]:
# ▶ 제공 코드 — 파서만 실행 (모델을 다시 호출하지 않음)
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
text = parser.invoke(ai_message)
print("종류:", type(text).__name__)
print(text)

종류: TextAccessor
완료: A라인 온도 센서 교체
예정: 14시 재가동
확인: 재가동 후 30분 동안 온도 로그 확인


### 6. 파이프(`|`)로 연결 — 체인
`프롬프트 | 모델 | 파서`는 앞 부품의 결과를 뒤 부품에 넘깁니다. 체인을 `invoke`할 때마다 모델도 다시 호출되므로 문장은 매번 조금 다를 수 있습니다. 비교할 것은 단계마다 결과의 **종류**입니다.

In [9]:
# ▶ 제공 코드 — 세 부품을 체인으로 연결
handover_chain = handover_prompt | llm | parser

result = handover_chain.invoke({"memo": memo})
print("종류:", type(result).__name__)
print(result)

종류: TextAccessor
완료: A라인 온도 센서 교체
예정: 14시 재가동
확인: 재가동 후 30분 동안 온도 로그 확인


### 7. 중간 출력 한눈에 보기
같은 메모가 지난 세 단계를 순서대로 나열합니다. 새 요청은 보내지 않습니다.

In [10]:
# ▶ 제공 코드 — 단계별 결과의 종류 (이미 받은 값만 출력)
for step, value in [("프롬프트 결과", prompt_value), ("모델 결과", ai_message), ("파서 결과", text)]:
    print(f"{step:8s} → {type(value).__name__}")
print("ChatPromptValue: 역할별 메시지를 담은 객체 → AIMessage: 모델이 반환한 답변 메시지 → str: 답변 본문 문자열")

프롬프트 결과  → ChatPromptValue
모델 결과    → AIMessage
파서 결과    → TextAccessor
ChatPromptValue: 역할별 메시지를 담은 객체 → AIMessage: 모델이 반환한 답변 메시지 → str: 답변 본문 문자열


### 정리
| 단계 | 결과의 종류 | 모델 호출 |
|---|---|---|
| 프롬프트 `handover_prompt.invoke(...)` | ChatPromptValue | 아직 안 함 |
| 모델 `llm.invoke(prompt_value)` | AIMessage | 1번 |
| 파서 `parser.invoke(ai_message)` | str | 안 함 |
| 체인 `handover_chain.invoke(...)` | str | 실행할 때마다 1번 |

**이어서 할 일** — 런박스 과제에서 같은 메모를 **작업자용 세 줄**과 **팀장용 한 줄**로 바꾸는 체인을 직접 만듭니다. 안내: `바이브코딩_대상별전달문/실습_대상별전달문_바이브코딩.html`